In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import seaborn as sns

# Hypothesis testing: Chi-Square Test within the Eniac case study

In this notebook we perform a chi-square test with the data from the Eniac case study, applying a post-hoc correction to perform pairwise tests and find the true winner.

## 1.&nbsp;State the Null Hypothesis and the Alternative Hypothesis.

## 2.&nbsp; Select an appropriate significance level alpha ($\alpha$).

It was decided that a relatively high alpha was acceptable in this case

## 3.&nbsp; Collect data that is random and independent

The important pieces of information (clicks on each element of interest & visits on each page) are scattered around. Let's collect them. Where are the .csv files? 🥸

In [2]:

# Element list eniac_a.csv
#url = 'https://drive.google.com/file/d/1x4DwDVc7uA2eIMEkmujmv81SfANjm6GC/view?usp=drive_link'
#path = 'https://drive.google.com/uc?export=download&id='+url.split('/')[-2]
eniac_a = pd.read_csv('data/eniac_a.csv')

# Element list eniac_b.csv
#url = 'https://drive.google.com/file/d/1Z4vLwetcGbk75cRihAp_rVtQ879vPYhu/view?usp=drive_link'
#path = 'https://drive.google.com/uc?export=download&id='+url.split('/')[-2]
eniac_b = pd.read_csv('data/eniac_b.csv')

# Element list eniac_c.csv
#url = 'https://drive.google.com/file/d/1mFk5UkygtUAmwx3KD3y8WRQVkF959YCv/view?usp=drive_link'
#path = 'https://drive.google.com/uc?export=download&id='+url.split('/')[-2]
eniac_c = pd.read_csv('data/eniac_c.csv')

# Element list eniac_d.csv
#url = 'https://drive.google.com/file/d/1e5ldQGnyQIL5UWYe1nDiKReNr1NGNvzg/view?usp=drive_link'
#path = 'https://drive.google.com/uc?export=download&id='+url.split('/')[-2]
eniac_d = pd.read_csv('data/eniac_d.csv')

In [55]:
eniac_a.head(3)   


,Element ID,Tag name,Name,No. clicks,Visible?,Snapshot information
0,48,h1,ENIAC,269,True,Homepage Version A - white SHOP NOW • http...
1,25,div,mySidebar,309,True,created 2021-09-14 • 14 days 0 hours 34 mi...
2,4,a,Mac,279,True,NaN


In [56]:
# eniac_a_version = eniac_a.iloc[0, 5]
# print(eniac_a_version)
# eniac_a_data = eniac_a.iloc[1, 5]
# print(eniac_a_data)
# pos = eniac_a.Name == 'SHOP NOW'
# eniac_a_clicks = eniac_a.loc[pos, ('No. clicks')]
# eniac_a_clicks

In [3]:
import re
import pandas as pd

# Extract visits from the info string
eniac_a_data = eniac_a.iloc[1, 5]
visits = int(re.search(r'([\d,]+) visits', eniac_a_data).group(1).replace(',', ''))
# Extract button name from the info string
eniac_a_version = eniac_a.iloc[0, 5]
button_name = re.search(r'-\s\w+\s([^•]+)', eniac_a_version).group(1).strip() #[0:8]

# Extract clicks for the button_name
pos = eniac_a.Name == button_name
clicks = eniac_a.loc[pos, 'No. clicks'].values[0]

# Build new df the order matters! clicks, then visits, or small to big!!!
results_df = pd.DataFrame({
    'metric': ['clicks', 'visits'],
    'A':   [clicks,  visits ]
})

print(results_df)

   metric      A
0  clicks    512
1  visits  25326


In [58]:
# button_name, visits

In [9]:


# Extract visits from the info string
eniac_b_data = eniac_b.iloc[1, 5]
visits = int(re.search(r'([\d,]+) visits', eniac_b_data).group(1).replace(',', ''))

eniac_b_version = eniac_b.iloc[0, 5]
button_name = re.search(r'-\s\w+\s([^•]+)', eniac_b_version).group(1).strip()#[0:8]


# Extract clicks for SHOP NOW
pos = eniac_b.Name == button_name
clicks = eniac_b.loc[pos, 'No. clicks'].values[0]

# Build new df
results_df['B']= pd.DataFrame({
    'B':  [clicks,  visits ]
})

# print(results_df)
# button_name

In [10]:
# Extract visits from the info string
eniac_c_data = eniac_c.iloc[1, 5]
visits = int(re.search(r'([\d,]+) visits', eniac_c_data).group(1).replace(',', ''))

eniac_c_version = eniac_c.iloc[0, 5]
button_name = re.search(r'-\s\w+\s([^•]+)', eniac_c_version).group(1).strip()#[0:9]


# Extract clicks for SHOP NOW
pos = eniac_c.Name == button_name
clicks = eniac_c.loc[pos, 'No. clicks'].values[0]

# Build new df
results_df['C']= pd.DataFrame({
    'C':  [clicks,  visits ]
})
# button_name
# print(results_df)

In [11]:
# Extract visits from the info string
eniac_d_data = eniac_d.iloc[1, 5]
visits = int(re.search(r'([\d,]+) visits', eniac_d_data).group(1).replace(',', ''))

eniac_d_version = eniac_d.iloc[0, 5]
button_name = re.search(r'-\s\w+\s([^•]+)', eniac_d_version).group(1).strip()#[0:9]


# Extract clicks for SHOP NOW
pos = eniac_d.Name == button_name
clicks = eniac_d.loc[pos, 'No. clicks'].values[0]

# Build new df
results_df['D']= pd.DataFrame({
    'D':  [clicks,  visits ]
})
button_name
print(results_df)

   metric      A      B      D      C
0  clicks    512    281    193    527
1  visits  25326  24747  25233  24876


In [62]:
# import re
# dfs = {'A': eniac_a, 'B': eniac_b, 'C': eniac_c, 'D': eniac_d}
# clicks = []
# visits = []
# for version, df in dfs.items():
#     c = df.iloc[21]['No. clicks']
#     snapshot = df.iloc[1]['Snapshot information']
#     v = int(re.search(r'(\d+) visits', snapshot).group(1))
#     clicks.append(c)
#     visits.append(v)
# no_clicks = [v - c for v, c in zip(visits, clicks)]
# ct1 = pd.DataFrame(
#     [clicks, no_clicks],
#     columns=['A', 'B', 'C', 'D'],
#     index=['Click', 'No-click']
# )
# print(ct1)

In [18]:
data = results_df.drop(columns='metric')
data.A.isin('512')

TypeError: only list-like objects are allowed to be passed to isin(), you passed a `str`

In [7]:
data 
diff_row = data.iloc[1] - data.iloc[0]
data = pd.concat([data, diff_row.to_frame().T], ignore_index=True)
data.drop(index=1)

,A,B,D
0,512,281,193
2,24814,24466,25040


## 4.&nbsp; Calculate the test result

In [92]:
import numpy as np
from scipy.stats import chi2
from scipy.stats import chi2_contingency
alpha = 0.001#  0.05
_, pvalue, _, _ = chi2_contingency(data)
if pvalue > alpha:
    print("The p-value is larger than alpha.")
    print("The click rate on each website (A, B, C and D) does not show any evidences for statistically significat differences.")

else:
    print("The p-value is smaller than alpha.")
    print("Alternative Hypothesis. There is evidence for statistically significat differences in the click rate for at least one of the websites compared to the others, either better or worse.")

pvalue

The p-value is smaller than alpha.
Alternative Hypothesis. There is evidence for statistically significat differences in the click rate for at least one of the websites compared to the others, either better or worse.


np.float64(6.833435052262316e-47)

## 5.&nbsp; Interpret the test result

## How do we decide who's the winner?

In [94]:
from itertools import combinations

versions = ['A', 'B', 'C', 'D' ]
pairs = list (combinations(versions,2))
pairs

bonf_alpha = 0.00005/len(pairs)

for v1, v2 in pairs:
    sub_table = data[[v1, v2]]
    #chi2_stat, p_val, dof, expected_loop = chi2_contingency(sub_table)
    chisq, p_val, dof, exp = chi2_contingency(sub_table)
    print(f"{v1} vs {v2} p = {p_val:.7}", 
    "significant" if p_val < bonf_alpha else "non_significant")
    

A vs B p = 7.19649e-15 significant
A vs C p = 0.4744331 non_significant
A vs D p = 2.276229e-32 significant
B vs C p = 2.328483e-17 significant
B vs D p = 2.81881e-05 non_significant
C vs D p = 6.479795e-36 significant


In [91]:
df = data 
diff_row = (df.iloc[1] - df.iloc[0]).to_frame().T
diff_row

,A,B,C,D
0,24814,24466,24349,25040
